In [ ]:
from theia.simulation.logging import LogLoader


loader = LogLoader("output.json")

In [ ]:
import itertools


pcl_detections = sorted(loader.blue_pcl_detections, key=lambda d: d.time)
for time, detections in itertools.groupby(pcl_detections, key=lambda d: d.time):
    detections = list(detections)
    if len(detections) >= 3:
        break

In [ ]:
import datetime


assert all([detection.target == detections[0].target for detection in detections])
detection = detections[0]
target = detection.target

t_next = detection.time + datetime.timedelta(seconds=1)
detections_next = [d for d in loader.blue_pcl_detections if d.time == t_next]
target_next = detections_next[0].target

In [ ]:
import numpy as np


np.array(target_next.point.as_tuple()) - np.array(target.point.as_tuple())

In [ ]:
from theia.mapping import pcl_detection_to_polygon


polygons = {
    f"Detection ID = {detection.detection_id}": pcl_detection_to_polygon(
        detection,
        target.alt,
        n_theta=360,
        n_phi=360,
    )
    for detection in detections
}
polygons2 = {
    f"Detection ID = {detection.detection_id}": pcl_detection_to_polygon(
        detection,
        target_next.alt,
        n_theta=360,
        n_phi=360,
    )
    for detection in detections_next
}
polygons.update(polygons2)

In [ ]:
from theia.mapping import RadarMap


map = RadarMap(
    sensors={f"ID {sensor.id}": sensor for sensor in loader.blue_pcl_sensors},
    targets={"target": target, "target_next": target_next},
    polygons=polygons,
).to_map()
map.location = (detection.sensor.receiver.lat, detection.sensor.receiver.lon)
map

In [ ]:
from theia.detection.pcl import PclDetector
from theia.test_data import load_pcl_example

sensors, trajcetories, grid = load_pcl_example()
trajectory = trajcetories[0]
RCS = trajectory.cross_section_model.rcs

In [ ]:
grid.model_dump_json()

In [ ]:
from typing import Any, Literal

import pydantic
import shapely

import theia
from theia.detection.pcl import pcl_track_init_update_masks
from theia.grids import LatLonHeightGrid
from theia.types import Sensor
from theia.util import mask_to_polygon


class GeoJSONPolygon(pydantic.BaseModel):
    type: Literal["Polygon"] = "Polygon"
    coordinates: list[list[list[float]]]

    @classmethod
    def from_shapely(cls, polygon: shapely.Polygon) -> "GeoJSONPolygon":
        geojson = shapely.geometry.mapping(polygon)
        return cls(
            coordinates=[list(map(list, ring)) for ring in geojson["coordinates"]]
        )


class GeoJSONMultiPolygon(pydantic.BaseModel):
    type: Literal["MultiPolygon"] = "MultiPolygon"
    # One extra nesting level: [polygon][ring][point][coordinate]
    coordinates: list[list[list[list[float]]]]

    @classmethod
    def from_shapely(cls, multi: shapely.MultiPolygon) -> "GeoJSONMultiPolygon":
        geojson = shapely.geometry.mapping(multi)
        return cls(
            coordinates=[
                [list(map(list, ring)) for ring in polygon]
                for polygon in geojson["coordinates"]
            ]
        )


GeoJSONGeometry = GeoJSONPolygon | GeoJSONMultiPolygon


class GeoJSONFeature(pydantic.BaseModel):
    type: str = "Feature"
    geometry: GeoJSONGeometry = pydantic.Field(discriminator="type")
    properties: dict[str, Any] = {}

    @classmethod
    def from_shapely(
        cls,
        shape: shapely.Polygon | shapely.MultiPolygon,
        properties: dict[str, Any] = {},
    ) -> "GeoJSONFeature":
        if isinstance(shape, shapely.Polygon):
            geometry = GeoJSONPolygon.from_shapely(shape)
        elif isinstance(shape, shapely.MultiPolygon):
            geometry = GeoJSONMultiPolygon.from_shapely(shape)
        else:
            raise TypeError(f"Unsupported geometry type: {type(shape)}")
        return cls(geometry=geometry, properties=properties)


def calculate_pcl_coverage(
    sensors: list[Sensor],
    grid: LatLonHeightGrid,
    rcs: float,
    snr_threshold: float = theia.config.SNR_THRESHOLD_PCL,
    doppler_threshold: float = theia.config.DOPPLER_SHIFT_THRESHOLD_PCL,
    delay_threshold: float = theia.config.DELAY_THRESHOLD_PCL,
) -> tuple[GeoJSONFeature, GeoJSONFeature]:
    """
    Calculate PCL coverage.

    Parameters
    ----------
    sensor: Sensor
        Sensor
    grid: LatLonHeightGrid
        Calculation grid
    rcs: float
        Radar cross section for which to calculate the coverage
    snr_threshold: float, default theia.config.SNR_THRESHOLD_PCL
        Minimum detectable threshold [dB]
    doppler_threshold: float, default theia.config.DOPPLER_SHIFT_THRESHOLD_PCL
        Minimum detectable Doppler shift [Hz]
    delay_threshold: float, default theia.config.DELAY_THRESHOLD_PCL
        Delay threshold for PCL [us].
        This is used to judge whether a given transmitter - target - receiver geometry
        is in the bistatic or the forward scattering regime.

    Returns
    -------
    track_init_coverage: GeoJSONFeature
        Region in which a track init can happen only using PCL
    track_update_coverage: GeoJSONFeature
        Region in which a track update can happen only using PCL
    """
    assert grid.altitude_values.shape[0] == 1

    detector = PclDetector(
        snr_threshold=snr_threshold,
        doppler_threshold=doppler_threshold,
        delay_threshold=delay_threshold,
    )

    track_init_mask, track_update_mask = pcl_track_init_update_masks(
        detector,
        sensors,
        grid,
        rcs,
    )

    polygons_init = mask_to_polygon(
        track_init_mask[:, :, 0],
        grid.latitude_values[0],
        grid.latitude_values[1] - grid.latitude_values[0],
        grid.longitude_values[0],
        grid.longitude_values[1] - grid.longitude_values[0],
    )
    import json

    # with open("polygons.json", "w") as file:
    #     json.dump(polygons_init, file)

    polygons_update = mask_to_polygon(
        track_update_mask[:, :, 0],
        grid.latitude_values[0],
        grid.latitude_values[1] - grid.latitude_values[0],
        grid.longitude_values[0],
        grid.longitude_values[1] - grid.longitude_values[0],
    )
    return (
        GeoJSONFeature(
            geometry=GeoJSONMultiPolygon.from_shapely(
                shapely.MultiPolygon(polygons_init)
            )
        ),
        GeoJSONFeature(
            geometry=GeoJSONMultiPolygon.from_shapely(
                shapely.MultiPolygon(polygons_update)
            )
        ),
    )


track_init, track_update = calculate_pcl_coverage([sensors[0]], grid, 1.0)

In [ ]:
from pydantic import TypeAdapter
from theia.types import Sensor

TypeAdapter(list[Sensor]).dump_json(sensors)

In [ ]:
sensors[0].model_dump_json()

In [ ]:
import cProfile

from theia.detection.pcl import pcl_track_init_update_masks

detector = PclDetector()

profiler = cProfile.Profile()
profiler.enable()

track_init_mask, track_update_mask = pcl_track_init_update_masks(
    detector,
    sensors,
    grid,
    RCS,
)

profiler.disable()
profiler.dump_stats("profile__pcl_coverage.prof")

In [ ]:
track_init_mask.shape

In [ ]:
grid.model_dump_json()

In [ ]:
import folium

from theia.mapping import RadarMap
from theia.util import mask_to_polygon


map = RadarMap(
    sensors={f"Sensor {sensor.id}": sensor for sensor in sensors},
    trajectories={"Target": trajectory},
).to_map()
map.location = (sensors[0].receiver.lat, sensors[0].receiver.lon)

polygons = mask_to_polygon(
    track_init_mask[:, :, 0],
    grid.latitude_values[0],
    grid.latitude_values[1] - grid.latitude_values[0],
    grid.longitude_values[0],
    grid.longitude_values[1] - grid.longitude_values[0],
)
for polygon in polygons:
    folium.GeoJson(polygon, fillColor="red", color="red").add_to(map)

polygons = mask_to_polygon(
    track_update_mask[:, :, 0],
    grid.latitude_values[0],
    grid.latitude_values[1] - grid.latitude_values[0],
    grid.longitude_values[0],
    grid.longitude_values[1] - grid.longitude_values[0],
)
for polygon in polygons:
    folium.GeoJson(polygon, fillColor="blue", color="blue").add_to(map)
map

In [ ]:
import numpy as np

from theia.detection.pcl import PclDetector
from theia.export_paraview import ParaviewExporter, PointOfInterest


pois: list[PointOfInterest] = []
poi_id = 0
for sensor in pcl_sensors:
    pois.append(
        PointOfInterest(
            id=poi_id,
            label=f"Rx {sensor.receiver.id}",
            type="Rx",
            lat=sensor.receiver.lat,
            lon=sensor.receiver.lon,
            alt=sensor.receiver.alt,
        )
    )
    poi_id += 1
    pois.append(
        PointOfInterest(
            id=poi_id,
            label=f"Tx {sensor.transmitter.id}",
            type="Tx",
            lat=sensor.transmitter.lat,
            lon=sensor.transmitter.lon,
            alt=sensor.transmitter.alt,
        )
    )
    poi_id += 1

exporter = ParaviewExporter(
    lat_min,
    lat_max,
    lat_res,
    lon_min,
    lon_max,
    lon_res,
)

sensor = pcl_sensors[0]

detector = PclDetector()

exporter.export_terrain(
    "test",
    # {
    #     "min detectable RCS": lambda *p: detector.minimum_detectable_rcs_vector(
    #         sensor.receiver, sensor.transmitter, np.array(p).reshape((1, 3))
    #     )
    # },
)
exporter.export_pois(pois, "pois.csv")